# SMS Spam Detection

This notebook documents the data science process for a simple spam detector using TF-IDF and Logistic Regression. The goal is to build a reproducible model that can be used by the Streamlit app without data leakage.

The EDA focuses on three questions:

- Is the dataset clean enough to model safely?
- What observable SMS patterns separate `spam` from `ham`?
- Which modeling safeguards are needed because of class imbalance and duplicated text?

## 1. Problem Definition

The task consists of classifying SMS messages as `ham` or `spam`. Since spam is usually the minority class, accuracy alone is not enough. The evaluation focuses on spam precision, spam recall, spam F1, ROC-AUC, average precision, and the confusion matrix.

For this use case, false negatives and false positives have different costs. A false negative lets spam reach the user, while a false positive can hide a legitimate message. The model therefore needs balanced attention to spam recall and spam precision.

In [ ]:
from pathlib import Path
import re
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

from app.huggingface_utils import load_sms_test_dataset
from app.model_utils import DATA_PATH, load_sms_dataset, train_model

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_colwidth', 120)


## 2. Load and Clean Data

The original CSV has useful columns `v1` and `v2`, plus empty unnamed columns. The cleaning step keeps only the label and message, normalizes labels, removes empty messages, encodes the target, and removes duplicated SMS messages before splitting to reduce leakage risk.

Before modeling, it is useful to compare the raw file against the cleaned dataset so the amount of discarded data is explicit.

In [ ]:
raw_df = pd.read_csv(DATA_PATH, encoding='latin-1')
df = load_sms_dataset(DATA_PATH)

quality_report = pd.DataFrame(
    {
        'metric': [
            'raw_rows',
            'raw_columns',
            'cleaned_rows',
            'removed_rows',
            'empty_messages_raw',
            'duplicate_label_message_pairs_raw',
            'duplicate_messages_raw',
            'duplicate_messages_cleaned',
            'missing_labels_raw',
            'missing_messages_raw',
        ],
        'value': [
            len(raw_df),
            raw_df.shape[1],
            len(df),
            len(raw_df) - len(df),
            int(raw_df['v2'].astype(str).str.strip().eq('').sum()),
            int(raw_df.duplicated(subset=['v1', 'v2']).sum()),
            int(raw_df.duplicated(subset=['v2']).sum()),
            int(df.duplicated(subset=['message']).sum()),
            int(raw_df['v1'].isna().sum()),
            int(raw_df['v2'].isna().sum()),
        ],
    }
)
quality_report


In [ ]:
df.head()


In [ ]:
df.info()
df['label'].value_counts()


## 3. Exploratory Data Analysis

Useful EDA for this problem checks class balance, message length, word patterns, and simple SMS-specific signals such as digits, URLs, currency symbols, exclamation marks, and urgent promotional vocabulary.

These signals are not used as separate manual features in the current model, but they help explain why TF-IDF n-grams and a linear classifier are a reasonable baseline.

In [ ]:
balance = df['label'].value_counts().to_frame('count')
balance['percent'] = (balance['count'] / len(df) * 100).round(2)
balance


In [ ]:
eda_df = df.copy()
eda_df['message_length'] = eda_df['message'].str.len()
eda_df['word_count'] = eda_df['message'].str.split().str.len()
eda_df['digit_count'] = eda_df['message'].str.count(r'\d')
eda_df['uppercase_count'] = eda_df['message'].str.count(r'[A-Z]')
eda_df['exclamation_count'] = eda_df['message'].str.count('!')
eda_df['question_count'] = eda_df['message'].str.count(r'\?')
eda_df['currency_symbol_count'] = eda_df['message'].str.count(r'[?$?]')
eda_df['has_url'] = eda_df['message'].str.contains(r'http|www\.|\.com|\.net', flags=re.IGNORECASE, regex=True)
eda_df['has_phone_like_number'] = eda_df['message'].str.contains(r'\d{5,}', regex=True)
eda_df['has_spam_keyword'] = eda_df['message'].str.contains(
    r'free|win|winner|prize|claim|urgent|call now|txt|reply|cash|award',
    flags=re.IGNORECASE,
    regex=True,
)

numeric_features = [
    'message_length',
    'word_count',
    'digit_count',
    'uppercase_count',
    'exclamation_count',
    'question_count',
    'currency_symbol_count',
]
eda_df.groupby('label')[numeric_features].describe().round(2)


In [ ]:
binary_features = ['has_url', 'has_phone_like_number', 'has_spam_keyword']
(
    eda_df.groupby('label')[binary_features]
    .mean()
    .mul(100)
    .round(2)
    .rename(columns=lambda col: f'{col}_percent')
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=eda_df, x='label', ax=axes[0])
axes[0].set_title('Class distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Messages')

sns.boxplot(data=eda_df, x='label', y='message_length', ax=axes[1])
axes[1].set_title('Message length by class')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Characters')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=eda_df, x='word_count', hue='label', bins=35, stat='density', common_norm=False, ax=axes[0])
axes[0].set_title('Word-count distribution')
axes[0].set_xlabel('Words')

sns.histplot(data=eda_df, x='digit_count', hue='label', bins=25, stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Digit-count distribution')
axes[1].set_xlabel('Digits')

plt.tight_layout()
plt.show()


### 3.1 Frequent Terms and N-Grams

N-grams show which tokens the model is likely to find useful. This view should not be used to manually tune on the test set; it is an interpretability aid for the full cleaned corpus.

In [ ]:
def top_ngrams(messages, ngram_range=(1, 1), top_n=15):
    vectorizer = CountVectorizer(
        lowercase=True,
        strip_accents='unicode',
        stop_words='english',
        ngram_range=ngram_range,
        min_df=2,
    )
    matrix = vectorizer.fit_transform(messages)
    counts = matrix.sum(axis=0).A1
    terms = vectorizer.get_feature_names_out()
    return (
        pd.DataFrame({'term': terms, 'count': counts})
        .sort_values('count', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

spam_unigrams = top_ngrams(eda_df.loc[eda_df['label'] == 'spam', 'message'], (1, 1))
ham_unigrams = top_ngrams(eda_df.loc[eda_df['label'] == 'ham', 'message'], (1, 1))

pd.concat(
    {
        'spam_top_unigrams': spam_unigrams,
        'ham_top_unigrams': ham_unigrams,
    },
    axis=1,
)


In [ ]:
spam_bigrams = top_ngrams(eda_df.loc[eda_df['label'] == 'spam', 'message'], (2, 2))
ham_bigrams = top_ngrams(eda_df.loc[eda_df['label'] == 'ham', 'message'], (2, 2))

pd.concat(
    {
        'spam_top_bigrams': spam_bigrams,
        'ham_top_bigrams': ham_bigrams,
    },
    axis=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=spam_bigrams, y='term', x='count', ax=axes[0], color='#ef6f6c')
axes[0].set_title('Top spam bigrams')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('')

sns.barplot(data=ham_bigrams, y='term', x='count', ax=axes[1], color='#4d96d7')
axes[1].set_title('Top ham bigrams')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


### 3.2 Representative Examples

Inspecting short examples with strong spam indicators helps check whether the dataset matches the real messages the app is expected to classify.

In [ ]:
eda_df.loc[
    eda_df['label'].eq('spam'),
    ['message', 'message_length', 'digit_count', 'currency_symbol_count', 'has_url', 'has_phone_like_number'],
].sort_values(['has_url', 'digit_count', 'message_length'], ascending=False).head(8)


In [ ]:
eda_df.loc[
    eda_df['label'].eq('ham'),
    ['message', 'message_length', 'digit_count', 'has_url', 'has_phone_like_number'],
].sample(8, random_state=42)


### 3.3 External SMS Collection Check

The project includes `sms+spam+collection/SMSSpamCollection` as an external validation fixture, not as training data. Comparing its balance with `spam.csv` helps verify that the functional tests exercise a similar but separately loaded source.

In [ ]:
external_df = load_sms_test_dataset()
external_balance = external_df['label'].value_counts().to_frame('external_count')
external_balance['external_percent'] = (external_balance['external_count'] / len(external_df) * 100).round(2)

training_balance = balance.rename(columns={'count': 'training_count', 'percent': 'training_percent'})
training_balance.join(external_balance)


## 4. Modeling Strategy

To avoid data leakage, the train/test split is done before fitting TF-IDF. TF-IDF and Logistic Regression are wrapped in one `Pipeline`, so vocabulary learning happens only on training folds. `GridSearchCV` is applied only on the training data with stratified folds. To reduce overfitting, the model uses TF-IDF document-frequency limits, n-gram validation, regularized Logistic Regression, and cross-validation.

EDA findings support three safeguards used by the training code:

- The target is imbalanced, so the split is stratified and Logistic Regression uses `class_weight='balanced'`.
- Duplicated messages are removed before splitting to reduce leakage across train/test.
- Spam contains repeated token patterns, so TF-IDF with unigrams/bigrams is a strong transparent baseline.

In [ ]:
metrics = train_model()
metrics['best_params'], metrics['cv_best_f1']


## 5. Final Test Metrics

The final metrics below are calculated on the holdout test set, which was not used to fit TF-IDF or choose hyperparameters.

In [ ]:
metrics['test_metrics']


In [ ]:
metrics['confusion_matrix']


## 6. Inference

The Streamlit app loads the persisted pipeline from `models/spam_model.joblib`, so inference uses the same trained TF-IDF vocabulary and Logistic Regression model.